In [82]:
import duckdb
import polars as pl
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


In [83]:
df.head()

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""teen suicide""","""a63a5c27-aae4-40b3-86c7-527496…","""new strategies for telemarketi…","""60d636ba-3ba8-439c-ae47-103837…","""coyote (2015-2021)""","""75ed55f6-5b9b-4de1-ac0a-0edb0e…",1660647868,2022-08-16 11:04:28
"""starflyer 59""","""250c38d6-0353-4c22-a371-f305d9…","""starflyer 59""","""320e6f9c-a990-4e7c-bc9c-807d30…","""she only knows""","""3f96450d-0ae9-353b-9f2d-5dd5c0…",1713954271,2024-04-24 10:24:31
"""joji""","""8264722b-df00-467a-858e-5c97cd…","""nectar""","""06549b4a-3c5a-43be-9819-82a15e…","""like you do""","""44ba7a0d-b10d-4ca9-be6c-f67c00…",1675962711,2023-02-09 17:11:51
"""thaiboy digital""","""68d311c0-525f-4f72-a044-84e545…","""back 2 life""","""d5851747-53b3-46f9-81b7-bf86bc…","""the kingdom""","""""",1668783321,2022-11-18 14:55:21
"""death grips""","""f9133036-ab3d-4e97-bd11-7a2c98…","""the money store""","""015ba8a6-97a9-4ad4-9ca4-382031…","""system blower""","""346d521f-5027-4a35-9276-5148fb…",1652449091,2022-05-13 13:38:11


## Most played songs and albums

"Album" is used loosely here. We use the definition of what the last.fm users has submitted as an album.

In [84]:
# Albums with most tracks played from
( 
    df.drop_nulls(pl.col("album_name"))
    .group_by(pl.col("album_name"))
    .agg(
        artist_name = pl.col("artist_name").first(),
        scrobbles_from_album = pl.len()
    )
).sort("scrobbles_from_album", descending=True)

album_name,artist_name,scrobbles_from_album
str,str,u32
"""dc snuff film / waste yrself""","""teen suicide""",767
"""icedancer""","""bladee""",714
"""the fool""","""bladee""",618
"""two star & the dream police""","""mk.gee""",574
"""april mixtape 2""","""snow strippers""",492
…,…,…
"""dc""","""dom corleo""",1
"""promises and lies""","""ub40""",1
"""sushi""","""james ferraro""",1


In [85]:
# Most played tracks
( 
    df
    .group_by(pl.col("track_name"))
    .agg(
        artist_name = pl.col("artist_name").first(),
        track_scrobbles = pl.len()
    )
).sort("track_scrobbles", descending=True).head(10)

track_name,artist_name,track_scrobbles
str,str,u32
"""egobaby""","""bladee""",193
"""girls just want to have fun""","""bladee""",156
"""blood""","""julia brown""",142
"""everything is fine""","""teen suicide""",139
"""hennessy & sailor moon""","""yung lean""",132
"""alesis""","""mk.gee""",127
"""western union""","""ecco2k""",126
"""som jag""","""dj billybool""",121
"""nostylist""","""destroy lonely""",120


## One-hit wonders — artists where you only played one track

In [86]:
( 
    df
    .group_by(pl.col("artist_name"))
    .agg(
        tracks = pl.col("track_name").unique().first(),
        total_artist_scrobbles = pl.len(),
        unique_tracks = pl.col("track_name").unique().len()
    )
    .filter(pl.col("unique_tracks") == 1)
).sort("total_artist_scrobbles", descending=True)

artist_name,tracks,total_artist_scrobbles,unique_tracks
str,str,u32,u32
"""w1nrar""","""m1ss1on""",51,1
"""contours""","""loose wood - ross from friends…",45,1
"""triplebeatscale""","""ravezz'97 mix""",44,1
"""oscar blesson""","""dum dum shawty! (feat. tyr)""",41,1
"""manchester orchestra""","""the gold - phoebe bridgers ver…",37,1
…,…,…,…
"""frédéric schubert""","""l'autre valse d'amélie""",1,1
"""axyom""","""bonestash""",1,1
"""dj class""","""#luvmiirite""",1,1


In [87]:
(
    df
    .with_columns(
        unique_track_count = pl.col("track_name").unique().len().over("artist_name")
    )
    .filter(pl.col("unique_track_count") == 1)
    .group_by(pl.col("artist_name"))
    .agg(
        tracks = pl.col("track_name").unique().first(),
        total_artist_scrobbles = pl.len(),
    )
    .sort("total_artist_scrobbles", descending=True)
).filter(pl.col("total_artist_scrobbles") > 1)

artist_name,tracks,total_artist_scrobbles
str,str,u32
"""w1nrar""","""m1ss1on""",51
"""contours""","""loose wood - ross from friends…",45
"""triplebeatscale""","""ravezz'97 mix""",44
"""oscar blesson""","""dum dum shawty! (feat. tyr)""",41
"""lostboyjay""","""could be wrong""",37
…,…,…
"""epic soundtracks""","""i wanna be free (as the lizard…",2
"""lionel hampton""","""a taste of honey""",2
"""hold""","""u""",2


## Song obsession tracking — songs with sudden play spikes

### Yearly framework on song-level

In [88]:
year = 2025
# First I want to get the most scrobbled tracks
(
    df
    .filter(pl.col("track_played_utc").dt.year() == year) 
    .group_by("track_name")
    .agg(total_scrobbles = pl.len())
).sort(pl.col("total_scrobbles"), descending=True).head(20)


track_name,total_scrobbles
str,u32
"""5""",59
"""mona lisa""",49
"""flash""",47
"""react [r tyler edit]""",46
"""bluey vuitton""",44
…,…
"""dash snow""",38
"""tonight""",38
"""nausicaä (love will be reveale…",37


#### Plotting when the songs are played

##### Cumulative count over the tracks to plot listening over time

In [89]:
# Cumulative listening per track, so each record gets a number representing the n'th time of listening to a track.
bv = (
    df.sort(pl.col("date_played_unix"), descending=False)
    .filter(pl.col("track_played_utc").dt.year() == year)
    .filter(pl.col("artist_name") == "bassvictim")
    .filter(pl.col("track_name").len().over("track_name") > 10)
    .with_columns(
        track_nth_scrobble = pl.col("track_name").cum_count().over(pl.col("track_name"))
    )
)
bv.head()

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,track_nth_scrobble
str,str,str,str,str,str,i64,datetime[μs],u32
"""bassvictim""","""""","""curse is lifted (club rmx)""","""""","""curse is lifted - club rmx""","""""",1737365548,2025-01-20 09:32:28,1
"""bassvictim""","""""","""curse is lifted (club rmx)""","""""","""curse is lifted - club rmx""","""""",1737552774,2025-01-22 13:32:54,2
"""bassvictim""","""""","""basspunk 2""","""""","""forever salty""","""""",1738433986,2025-02-01 18:19:46,1
"""bassvictim""","""""","""basspunk 2""","""""","""i'll be there""","""""",1738434337,2025-02-01 18:25:37,1
"""bassvictim""","""""","""basspunk 2""","""""","""hum it to google""","""""",1738434575,2025-02-01 18:29:35,1


In [90]:
import plotly.express as px
px.line(bv, "track_played_utc", "track_nth_scrobble", color="track_name", title="My habits of my top Bassvictim tracks over time (2025)")

### Yearly framework on artist-level

In [91]:
year = 2025 
relevant_artists_25 = (
    df
    .sort(pl.col("date_played_unix"), descending=False)
    .filter(pl.col("track_played_utc").dt.year() == year)
    .group_by(pl.col("artist_name"))
    .agg(scrobbles_a_year = pl.len())
).sort(pl.col("scrobbles_a_year"), descending=True).head(5)
relevant_artists_25

artist_name,scrobbles_a_year
str,u32
"""bladee""",459
"""dean blunt""",411
"""bassvictim""",369
"""astrid sonne""",301
"""vegyn""",301


In [92]:
# Cumulative listening per artist, so each artist gets a number representing the n'th time of listening.
artists_per_week = (
    df
    .join(relevant_artists_25, "artist_name", "semi")
    .sort(pl.col("date_played_unix"), descending=False)
    .filter(pl.col("track_played_utc").dt.year() == year)
    .with_columns(
        week = pl.col("track_played_utc").dt.ordinal_day(),
    )
    .group_by(pl.col("artist_name"), pl.col("week"))
    .agg(
        num_scrobbles = pl.len()
    )
    # .with_columns(
    #     track_nth_scrobble = pl.col("track_name").cum_count().over(pl.col("track_name"))
    # )
).sort(pl.col("week"), descending=False)
artists_per_week.head(5)

artist_name,week,num_scrobbles
str,i16,u32
"""bassvictim""",5,1
"""astrid sonne""",5,2
"""bladee""",5,1
"""bladee""",8,2
"""bassvictim""",10,1


In [93]:
from scipy.signal import savgol_filter
artists_per_week = artists_per_week.with_columns(
    filtered_scrobbles = savgol_filter(
        artists_per_week.select(pl.col("num_scrobbles")).to_numpy().flatten(),
        window_length=4,
        polyorder=3
        )
)
px.line(artists_per_week, "week", "filtered_scrobbles", color="artist_name")